# Variant D — CoT Trace Generation

Generates **~2,000 Chain-of-Thought reasoning traces** from e-SNLI training examples  
using `deepseek-ai/DeepSeek-R1-Distill-Qwen-7B` via the HuggingFace Inference API.

| | |
|---|---|
| **Target** | 667 traces × 3 labels = 2,001 total (balanced) |
| **Rate limit** | HF free tier ~500–1,000 req/day → takes 2–3 daily sessions |
| **Resume** | Re-run any time — skips already-completed rows automatically |

**Steps before running:**
1. Get a free HF token → https://huggingface.co/settings/tokens  
2. Edit **Cell 2** (config) with your token and Drive paths  
3. Run all cells

In [ ]:
# Cell 1 — Install dependencies
!pip install -q requests pandas

In [ ]:
# ══════════════════════════════════════════════════════════
# Cell 2 — CONFIGURATION  (only cell you need to edit)
# ══════════════════════════════════════════════════════════

# 1. Your HuggingFace API token
#    Get one free at: https://huggingface.co/settings/tokens
HF_TOKEN = "hf_PASTE_YOUR_TOKEN_HERE"

# 2. Paths on Google Drive
ESNLI_TRAIN_1 = "/content/drive/MyDrive/Old-Explanations-New-Rationales-Bridging-e-SNLI-and-LLM-Chain-of-Thought-in-Encoder-Based-NLI/Datasets/E-SNLI/esnli_train_1.csv"
ESNLI_TRAIN_2 = "/content/drive/MyDrive/Old-Explanations-New-Rationales-Bridging-e-SNLI-and-LLM-Chain-of-Thought-in-Encoder-Based-NLI/Datasets/E-SNLI/esnli_train_2.csv"
OUTPUT_PATH   = "/content/drive/MyDrive/Old-Explanations-New-Rationales-Bridging-e-SNLI-and-LLM-Chain-of-Thought-in-Encoder-Based-NLI/Datasets/CoT/cot_traces.csv"

# 3. Generation settings
MODEL_ID    = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"  # best open-source reasoning model
N_PER_LABEL = 667    # 667 × 3 labels = 2,001 traces total

# ══════════════════════════════════════════════════════════

assert HF_TOKEN != "hf_PASTE_YOUR_TOKEN_HERE", \
    "Paste your HuggingFace token in HF_TOKEN above!"

print(f"Model:   {MODEL_ID}")
print(f"Target:  {N_PER_LABEL * 3:,} traces ({N_PER_LABEL} per label)")
print(f"Output:  {OUTPUT_PATH}")
print(f"Token:   {HF_TOKEN[:8]}...{HF_TOKEN[-4:]}")

In [ ]:
# Cell 3 — Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Cell 4 — All helper functions (no external imports from project)
import os
import re
import time
from pathlib import Path
from typing import Optional, Set

import pandas as pd
import requests

# ── Constants ──────────────────────────────────────────────────────────────

HF_API_URL   = "https://api-inference.huggingface.co/v1/chat/completions"
VALID_LABELS = {"entailment", "neutral", "contradiction"}

# Plain-English descriptions — injected instead of label words to prevent
# the MLM head from learning a shortcut from the label name.
LABEL_DESCRIPTIONS = {
    "entailment":    "the hypothesis logically follows from the premise",
    "neutral":       "the hypothesis is neither confirmed nor contradicted by the premise",
    "contradiction": "the hypothesis contradicts what the premise states",
}

PROMPT_TEMPLATE = """\
You are a logical reasoning assistant for Natural Language Inference.

Premise: {premise}
Hypothesis: {hypothesis}
Relationship: {label_description}

Write a chain-of-thought reasoning trace explaining WHY this relationship holds.

Rules:
- Exactly 3 to 5 sentences. Label each \"Step 1:\", \"Step 2:\", etc.
- Do NOT use the words \"entailment\", \"neutral\", or \"contradiction\".
- Do NOT copy the premise or hypothesis verbatim.
- Each step must make a distinct logical point.
- The final step states the conclusion.

Reasoning:"""


# ── Prompt builder ─────────────────────────────────────────────────────────

def build_prompt(premise: str, hypothesis: str, label: str) -> str:
    return PROMPT_TEMPLATE.format(
        premise=premise,
        hypothesis=hypothesis,
        label_description=LABEL_DESCRIPTIONS[label],
    )


# ── DeepSeek-R1 post-processing ────────────────────────────────────────────

def strip_think_tags(text: str) -> str:
    """Remove <think>...</think> blocks that DeepSeek-R1 adds internally."""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()


# ── HuggingFace Inference API ──────────────────────────────────────────────

def call_hf(prompt: str, max_retries: int = 6, timeout: int = 60) -> Optional[str]:
    """Call HF Inference API. Returns generated text or None.

    Retry behaviour:
      429 (rate limit) → sleep Retry-After seconds, retry
      503 (loading)    → sleep 30s, retry
      other error      → return None immediately
    """
    headers = {
        "Authorization": f"Bearer {HF_TOKEN}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": MODEL_ID,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 300,
        "temperature": 0.3,
        "stream": False,
    }

    for attempt in range(max_retries):
        try:
            resp = requests.post(HF_API_URL, headers=headers, json=payload, timeout=timeout)
        except requests.exceptions.Timeout:
            print(f"  [timeout — attempt {attempt + 1}]", flush=True)
            time.sleep(10)
            continue
        except requests.exceptions.RequestException as e:
            print(f"  [request error: {e}]", flush=True)
            return None

        if resp.status_code == 200:
            try:
                text = resp.json()["choices"][0]["message"]["content"]
                text = strip_think_tags(text)
                return text if len(text) >= 30 else None
            except (KeyError, IndexError, ValueError):
                return None

        elif resp.status_code == 429:
            wait = int(resp.headers.get("Retry-After", 60))
            print(f"\n  [Rate limit — sleeping {wait}s (attempt {attempt + 1}/{max_retries})]",
                  flush=True)
            time.sleep(wait)

        elif resp.status_code == 503:
            print(f"\n  [Model loading (503) — sleeping 30s (attempt {attempt + 1}/{max_retries})]",
                  flush=True)
            time.sleep(30)

        else:
            print(f"\n  [HTTP {resp.status_code}: {resp.text[:120]}]", flush=True)
            return None

    print(f"  [Max retries reached — skipping example]", flush=True)
    return None


# ── Data loading ───────────────────────────────────────────────────────────

def load_esnli() -> pd.DataFrame:
    """Load both e-SNLI training CSVs and concatenate."""
    dfs = []
    for path in (ESNLI_TRAIN_1, ESNLI_TRAIN_2):
        df = pd.read_csv(path, usecols=["gold_label", "Sentence1", "Sentence2", "Explanation_1"])
        dfs.append(df)
    df = pd.concat(dfs, ignore_index=True)
    df = df[df["gold_label"].isin(VALID_LABELS)]
    df = df.dropna(subset=["Sentence1", "Sentence2", "Explanation_1"])
    return df.reset_index(drop=True)


def make_subset(df: pd.DataFrame) -> pd.DataFrame:
    """Stratified sample: N_PER_LABEL rows per label, with a stable pair_id."""
    subset = (
        df.groupby("gold_label", group_keys=False)
          .apply(lambda g: g.sample(min(N_PER_LABEL, len(g)), random_state=42))
          .reset_index(drop=True)
    )
    subset.insert(0, "pair_id", range(len(subset)))
    return subset


# ── Generation loop ────────────────────────────────────────────────────────

def generate(source_df: pd.DataFrame, sleep_between: float = 1.5) -> None:
    """Generate CoT traces with checkpoint/resume.

    Appends one row per example immediately after generation.
    Re-running this cell skips already-completed rows.
    """
    out = Path(OUTPUT_PATH)
    out.parent.mkdir(parents=True, exist_ok=True)

    # Find already-completed pair_ids
    done: Set[int] = set()
    if out.exists():
        existing = pd.read_csv(out, usecols=["pair_id", "cot_rationale"], dtype={"pair_id": int})
        done = set(
            existing.loc[
                existing["cot_rationale"].notna() &
                (existing["cot_rationale"].str.strip() != ""),
                "pair_id",
            ]
        )
        print(f"Resuming — {len(done):,} done, {len(source_df) - len(done):,} remaining.\n")

    remaining  = source_df[~source_df["pair_id"].isin(done)].reset_index(drop=True)
    total      = len(source_df)
    done_count = len(done)

    if remaining.empty:
        print("All traces already generated — run the Validate cell to inspect.")
        return

    file_exists = out.exists()
    start_time  = time.time()

    for i, row in enumerate(remaining.itertuples(index=False)):
        prompt = build_prompt(row.Sentence1, row.Sentence2, row.gold_label)
        cot    = call_hf(prompt)

        pd.DataFrame([{
            "pair_id":       row.pair_id,
            "gold_label":    row.gold_label,
            "Sentence1":     row.Sentence1,
            "Sentence2":     row.Sentence2,
            "Explanation_1": row.Explanation_1,
            "cot_rationale": cot if cot is not None else "",
            "cot_model":     MODEL_ID,
        }]).to_csv(out, mode="a", header=not file_exists, index=False)

        file_exists = True
        done_count += 1

        if (i + 1) % 50 == 0:
            elapsed = (time.time() - start_time) / 60
            rate    = (i + 1) / elapsed if elapsed > 0 else 0
            eta     = (total - done_count) / rate if rate > 0 else float("inf")
            print(f"[{done_count:,}/{total:,}] "
                  f"elapsed={elapsed:.1f}min | "
                  f"rate={rate:.1f} ex/min | "
                  f"ETA=~{eta:.0f}min", flush=True)

        time.sleep(sleep_between)

    elapsed_total = (time.time() - start_time) / 60
    print(f"\nSession done — {done_count:,}/{total:,} traces in {elapsed_total:.1f} min.")
    if done_count < total:
        print("Rate limit hit — run this notebook again tomorrow to continue.")


# ── Validation ─────────────────────────────────────────────────────────────

def validate() -> None:
    """Print a quality report for the current cot_traces.csv."""
    path = Path(OUTPUT_PATH)
    if not path.exists():
        print(f"File not found: {path}")
        return

    df        = pd.read_csv(path, dtype={"pair_id": int})
    total     = len(df)
    non_empty = df["cot_rationale"].notna() & (df["cot_rationale"].str.strip() != "")
    filled    = non_empty.sum()
    leakage   = df.loc[non_empty, "cot_rationale"].str.lower().apply(
        lambda t: any(w in t for w in ["entailment", "neutral", "contradiction"])
    ).sum()
    wc = df.loc[non_empty, "cot_rationale"].str.split().apply(len)

    print("═" * 47)
    print("  CoT Trace Validation")
    print("═" * 47)
    print(f"  Total rows:          {total:>6,}  / {N_PER_LABEL * 3:,} target")
    print(f"  Completed:           {filled:>6,}  ({filled / (N_PER_LABEL * 3) * 100:.1f}%)")
    print(f"  Label word leakage:  {leakage:>6,}  ({leakage / max(filled,1) * 100:.1f}%)  ← target <5%")
    print(f"  Avg word count:      {wc.mean():>6.1f}")
    print(f"  Min / Max words:     {wc.min():>3} / {wc.max()}")
    print("  Label distribution:")
    for label, count in df["gold_label"].value_counts().sort_index().items():
        print(f"    {label:<15} {count:>5,}  ({count / total * 100:.1f}%)")
    print("═" * 47)


print("✓ All functions loaded.")

In [ ]:
# Cell 5 — Load e-SNLI and build subset
print("Loading e-SNLI training data...")
esnli_df = load_esnli()
print(f"  Loaded {len(esnli_df):,} examples")

print("Building stratified subset...")
subset = make_subset(esnli_df)
print(f"  Subset: {len(subset):,} examples")
print(subset["gold_label"].value_counts().to_string())

In [ ]:
# Cell 6 — Check existing progress
from pathlib import Path
out = Path(OUTPUT_PATH)

if out.exists():
    df_check = pd.read_csv(out)
    done_mask = df_check["cot_rationale"].notna() & (df_check["cot_rationale"].str.strip() != "")
    print(f"Existing file: {out}")
    print(f"  Rows total:     {len(df_check):,}")
    print(f"  Completed:      {done_mask.sum():,}")
    print(f"  Still needed:   {N_PER_LABEL * 3 - done_mask.sum():,}")
    print()
    print("Label breakdown so far:")
    print(df_check.loc[done_mask, "gold_label"].value_counts().to_string())
else:
    print("No existing file — will start fresh.")

In [ ]:
# Cell 7 — Quick API test (1 example — confirms token + model are working)
test_prompt = build_prompt(
    premise="A dog is running through a field.",
    hypothesis="An animal is outside.",
    label="entailment",
)

print("Testing HF API with one example...")
result = call_hf(test_prompt)

if result:
    print("✓ API working. Sample output:")
    print("-" * 55)
    print(result)
    print("-" * 55)
else:
    print("✗ API call failed.")
    print(f"  Token:  {HF_TOKEN[:8]}...")
    print(f"  Model:  {MODEL_ID}")
    print("  Fix the issue before running Cell 8.")

In [ ]:
# Cell 8 — Generate (or resume)
#
# Runs until the daily rate limit is hit, then stops gracefully.
# Re-run this notebook tomorrow — it picks up where it left off.
# Every example is saved to disk immediately.

generate(subset)

In [ ]:
# Cell 9 — Validate quality
validate()

In [ ]:
# Cell 10 — Preview one sample per label (manual quality check)
df_preview = pd.read_csv(OUTPUT_PATH)
done_mask  = df_preview["cot_rationale"].notna() & (df_preview["cot_rationale"].str.strip() != "")
samples    = df_preview[done_mask].groupby("gold_label").head(1).reset_index(drop=True)

for _, row in samples.iterrows():
    print(f"Label:       {row['gold_label']}")
    print(f"Premise:     {row['Sentence1']}")
    print(f"Hypothesis:  {row['Sentence2']}")
    print(f"Human expl:  {row['Explanation_1']}")
    print(f"CoT trace:")
    print(row["cot_rationale"])
    print("-" * 60)